# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

# 🌐 1. Environment Setup


In [ ]:
# ── Cell 1: 💢 Environment Sync & Cache Exorcism ────────────
!pip install -q -U bitsandbytes triton transformers datasets
import os, shutil, sys, gc, torch

# 1. Nuclear Sync: Ensure fresh library code
REPO_NAME = 'EFV-nn'
if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
!git clone -q https://github.com/ey3lock3r/EFV-nn.git
if f'/kaggle/working/{REPO_NAME}/src' not in sys.path: sys.path.insert(0, f'/kaggle/working/{REPO_NAME}/src')

# 2. Path Configuration
SAVE_PATH = '/kaggle/working/ppc_3b_pretrain.pt'

# 3. Corruption Purge & Cache Wipe
if os.path.exists(SAVE_PATH) and os.path.getsize(SAVE_PATH) < 5 * 1024**3:
    print(f'🗑️ Purging corrupted remnant ({os.path.getsize(SAVE_PATH)/1e9:.2f} GB)')
    os.remove(SAVE_PATH)
if os.path.exists(SAVE_PATH + '.tmp'): os.remove(SAVE_PATH + '.tmp')

for p in ['/tmp/torchinductor_root', os.path.expanduser('~/.cache/torch')]:
    if os.path.exists(p): shutil.rmtree(p)

# 4. Checkpoint Discovery

gc.collect(); torch.cuda.empty_cache()
# 4. Checkpoint Discovery (Main > Prev > Fresh)
if os.path.exists(SAVE_PATH) and os.path.getsize(SAVE_PATH) > 5 * 1024**3:
    LOAD_PATH = SAVE_PATH
else:
    candidates = [f for f in os.listdir('.') if f.endswith('.pt') and '_prev' in f]
    healthy = [f for f in candidates if os.path.getsize(f) > 5 * 1024**3]
    LOAD_PATH = max(healthy, key=os.path.getsize) if healthy else SAVE_PATH

print(f'✅ Environment Ready. Load Target: {LOAD_PATH}')


# 🔑 2. Secrets & Monitoring


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
import os, wandb
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    WANDB_KEY = secrets.get_secret('WANDB_API_KEY')
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print('✅ Logged into W&B and HF')
except Exception as e:
    print(f'⚠️ Secrets failure: {e}. Training will continue without W&B.')


# 📦 3. Data Pipeline


In [ ]:
# ── Cell 3: Data Pipeline (FineWeb-Edu Streaming) ────────────
from datasets import load_dataset
import torch

def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    def gen():
        buffer = []
        for ex in ds:
            buffer.extend(tokenizer(ex['text'])['input_ids'])
            while len(buffer) >= (batch_size * seq_len):
                yield torch.tensor(buffer[:batch_size * seq_len]).view(batch_size, seq_len)
                buffer = buffer[batch_size * seq_len:]
    return gen()


# 📡 4. W&B Rescue (Optional)


In [ ]:
# ── Cell 4: Optional W&B Rescue (Pull checkpoint from cloud) ─────────────────
# Run this cell AFTER Cell 2 (Secrets) to restore a checkpoint from a previous run.
import shutil, os, wandb, json as _json
RESCUE_RUN_PATH = "dhinsresearch/ppc-v3-automated/rn11y61i"
RESCUE_FILE = "ppc_3b_pretrain.pt"

def rescue_checkpoint(run_path, filename, target_path):
    print(f"Syncing {filename} from {run_path}...")

    # Guard: wandb.restore needs a valid API key. Fail fast with a clear message
    # rather than a cryptic CommError deep in the wandb internals.
    if not wandb.api.api_key:
        print("Rescue aborted: no wandb API key found. Run Cell 2 (Secrets) first.")
        return

    try:
        # replace=True: always pull the cloud version, even if a local copy exists.
        # Default replace=False would silently skip the download and return the stale local file.
        restored_file = wandb.restore(filename, run_path=run_path, replace=True, root='/kaggle/working')
        if restored_file is None:
            raise FileNotFoundError(f"{filename} not found in run {run_path}")

        source_path = os.path.abspath(restored_file.name)
        target_path = os.path.abspath(target_path)

        if source_path != target_path:
            if os.path.exists(target_path):
                print(f'Removing existing file at target: {target_path}')
                os.remove(target_path)
            shutil.move(source_path, target_path)

        print(f"Checkpoint rescued to {target_path} ({os.path.getsize(target_path)/1e9:.2f} GB)")

        # Also rescue the .meta sidecar if it exists in the run.
        # Without it, _wandb_id stays None and the next session creates a new wandb run,
        # fragmenting the loss curve dashboard.
        meta_filename = filename + '.meta'
        try:
            meta_file = wandb.restore(meta_filename, run_path=run_path, replace=True, root='/kaggle/working')
            if meta_file is not None:
                meta_source = os.path.abspath(meta_file.name)
                meta_target = target_path + '.meta'
                if meta_source != meta_target:
                    shutil.move(meta_source, meta_target)
                print(f"Sidecar rescued: wandb run continuity preserved.")
            else:
                # No sidecar in the run — write one now using the rescued run_path's run id.
                run_id = run_path.split('/')[-1]
                with open(target_path + '.meta', 'w') as f:
                    _json.dump({'wandb_id': run_id}, f)
                print(f"Sidecar created from run_path id: {run_id}")
        except Exception as e:
            print(f"Could not rescue sidecar (non-fatal): {e}")

        print("Rescue complete. Proceed to Cell 5.")

    except Exception as e:
        print(f"Rescue failed: {e}")

# Execute Rescue
rescue_checkpoint(RESCUE_RUN_PATH, RESCUE_FILE, SAVE_PATH)


# 🏗️ 5. Architectural Setup


In [ ]:
# ── Cell 5: Architectural Setup & v3 State Management ──
import torch, gc, os
from efv_nn.ppc_sharded import ShardedPPCGraphLLM
from transformers import AutoTokenizer

# 1. Config
VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
PRIME_DELAYS = [1, 2, 3, 5, 7, 11, 13, 17]

# Toggle Diagnostics (Set "1" to enable NaN sniffing)
os.environ["PPC_DEBUG"] = "0"

# 2. Model
print('Instantiating 3.2B PPC-OCNS v3...')
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', token=os.environ.get('HF_TOKEN'))
model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, prime_delays=PRIME_DELAYS)

# 3. Atomic State Resume
# LOAD_PATH is set by Cell 1 discovery logic. We auto-detect whether a valid checkpoint
# exists rather than requiring a manual RESUME flag that is easy to forget.
START_STEP, START_PHASE = 0, 0
OPT_STATE = None

_ckpt_exists = os.path.exists(LOAD_PATH) and os.path.getsize(LOAD_PATH) > 5 * 1024**3
if _ckpt_exists:
    print(f'Resuming from {LOAD_PATH}...')
    ckpt = torch.load(LOAD_PATH, map_location='cpu', mmap=True, weights_only=False)
    START_STEP  = ckpt.get('step', 0)
    START_PHASE = ckpt.get('phase', 0)
    model.load_state_dict(ckpt['model_state'], strict=False)

    # Pre-FSM checkpoint guard: checkpoints saved before the PilotFSM was introduced
    # always stored phase=0 regardless of actual convergence. Heuristic: if the run
    # has > 2000 steps but phase=0, the model has cleared Phase 0 — start at Phase 1
    # to prevent the FSM from immediately cascading through promotions on a warm model.
    if START_PHASE == 0 and START_STEP > 2000:
        START_PHASE = 1
        print(f'  Pre-FSM checkpoint detected (step={START_STEP}, stored phase=0). Bumping START_PHASE to 1.')
    OPT_STATE = ckpt.get('opt_state')

    # Restore base_local_lr for each layer — this is a plain Python attribute (not an
    # nn.Parameter) set during Phase Promotion, so it is NOT inside model_state.
    # Without this, all layers silently revert to Phase 0 local_lr=0.05 after resume,
    # causing the PPC contraction dynamics to regress regardless of the saved phase.
    _PHASE_LOCAL_LR = {0: 0.05, 1: 0.15, 2: 0.25, 3: 0.10}
    for layer in model.layers:
        layer.base_local_lr = _PHASE_LOCAL_LR[START_PHASE]
    print(f'  base_local_lr restored to {_PHASE_LOCAL_LR[START_PHASE]} for Phase {START_PHASE}')

    del ckpt; gc.collect()
    print(f'Ready. Step: {START_STEP} | Phase: {START_PHASE} | Params: {model.total_params/1e9:.2f}B')
else:
    print(f'No valid checkpoint found at {LOAD_PATH}. Starting fresh.')
    print(f'Ready. Step: 0 | Phase: 0 | Params: {model.total_params/1e9:.2f}B')


# 🔬 6. Model Identity Audit


In [ ]:
# ── Cell 6: 🔬 Model Identity Audit (Sanity Check) ──
def run_identity_audit(model, path):
    def _audit_weights(label, weights):
        w_mean = weights.abs().mean().item()
        w_std  = weights.std().item()
        status = "✅ LEARNED" if w_std > 0.05 else "❌ RANDOM/NOISE"
        print(f"[{label}] Mean: {w_mean:.6f} | Std: {w_std:.6f} | Status: {status}")
        return w_std > 0.05

    print("🔍 Auditing Memory vs Disk...")
    # 1. Memory
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    memory_learned = _audit_weights("MEMORY", raw_model.embedding.weight)
    
    # 2. Disk
    if os.path.exists(path):
        try:
            ckpt = torch.load(path, map_location='cpu', mmap=True)
            s_dict = ckpt['model_state'] if 'model_state' in ckpt else ckpt
            w_disk = s_dict.get('embedding.weight') or s_dict.get('_orig_mod.embedding.weight')
            if w_disk is not None:
                _audit_weights("DISK  ", w_disk.float())
            else:
                print("⚠️ DISK: Could not find 'embedding.weight' in checkpoint.")
        except Exception as e:
            print(f"⚠️ DISK: Failed to read checkpoint: {e}")
    else:
        print("⚪ DISK: No checkpoint found at target path.")
    
    if not memory_learned:
        print("\n❗ WARNING: Model in memory is RANDOM. Please verify your load logic.")

# 🚀 To perform a sanity check, uncomment the line below:
# run_identity_audit(model, LOAD_PATH)


# 🚀 7. Production Training Loop


In [ ]:
# ── Cell 7: Automated Pilot & Memory Guard ──
import time, wandb, gc, math, torch, os, itertools
import bitsandbytes.optim as bnb
import torch.nn.functional as F


class PilotFSM:
    """
    Finite-state machine for automated training surgery.

    States
    ------
    NOMINAL    — healthy training; monitors for transition signals
    DIVERGING  — energy spiralling; LR halved on entry, exits when trend normalises
    NOISE_WALL — high energy + flat loss; LR shocked up on entry, exits after cooldown
    PLATEAU    — converged energy + flat loss; spectral blend nudged on entry, exits after cooldown
    PROMOTING  — energy below phase target; phase advanced on entry, exits after cooldown

    Design rules
    ------------
    - Actions fire ONCE on state entry, not every step the condition holds.
    - Non-NOMINAL states block new surgeries (no stacking).
    - Each state has its own cooldown before returning to NOMINAL.
    - DIVERGING is the only emergency: it can interrupt any other state immediately.
    - Step guards use rolling buffer length so the FSM is resume-agnostic
      (works correctly whether start_step=0 or start_step=20000).
    """

    NOMINAL    = 'NOMINAL'
    DIVERGING  = 'DIVERGING'
    NOISE_WALL = 'NOISE_WALL'
    PLATEAU    = 'PLATEAU'
    PROMOTING  = 'PROMOTING'

    # Steps to remain in state before returning to NOMINAL for re-evaluation.
    # DIVERGING has no minimum — exits as soon as trend normalises.
    COOLDOWNS = {
        NOISE_WALL: 200,
        PLATEAU:    150,
        PROMOTING:  200,  # momentum stabilisation after LR change + rolling buffer refill
    }

    def __init__(self, model, opt, phases, lr_floor, lr_ceil, blend_cap):
        self.model      = model
        self.opt        = opt
        self.phases     = phases
        self.lr_floor   = lr_floor
        self.lr_ceil    = lr_ceil
        self.blend_cap  = blend_cap

        self.state            = self.NOMINAL
        self.state_entered_at = 0
        self.current_phase    = 0

    # ── helpers ──────────────────────────────────────────────────────────────

    def _base_lr(self):
        return self.opt.param_groups[0]['lr']

    def _set_lr(self, new_lr):
        for pg_idx, pg in enumerate(self.opt.param_groups):
            pg['lr'] = new_lr * (10 if pg_idx == 1 else 1)

    def _in_cooldown(self, step):
        return (step - self.state_entered_at) < self.COOLDOWNS.get(self.state, 0)

    def _enter(self, new_state, step):
        self.state            = new_state
        self.state_entered_at = step

    # ── main tick ────────────────────────────────────────────────────────────

    def tick(self, step, avg_e, e_trend, l_delta, rolling_e, rolling_l):
        """
        Called once per training step after rolling buffers have enough data.
        Returns (current_phase, action_name | None, wandb_log_dict).
        """
        action = None
        log    = {}

        # DIVERGING is the only emergency — interrupt any state.
        if e_trend > 1.5 or avg_e > 6000:
            if self.state != self.DIVERGING:
                new_lr = max(self.lr_floor, self._base_lr() * 0.5)
                self._set_lr(new_lr)
                rolling_e.clear(); rolling_l.clear()
                self._enter(self.DIVERGING, step)
                action = self.DIVERGING
                log    = {'surgery': 1, 'surgery_lr': new_lr, 'e_trend': e_trend}
                print(f'\nSURGERY A [{step}]: Diverging (e_trend:{e_trend:.2f} E:{avg_e:.1f}) LR->{new_lr:.2e}')
            # else: already handling it — no repeated action, just monitor
            return self.current_phase, action, log

        # Non-emergency: exit DIVERGING once trend normalises
        if self.state == self.DIVERGING:
            if e_trend < 1.2 and avg_e < 3000:
                self._enter(self.NOMINAL, step)
                print(f'\nRECOVERED [{step}]: Energy trend normalised. Returning to NOMINAL.')
            return self.current_phase, action, log

        # All other states: exit to NOMINAL after their cooldown, then re-evaluate next tick
        if self.state != self.NOMINAL:
            if not self._in_cooldown(step):
                self._enter(self.NOMINAL, step)
            return self.current_phase, action, log

        # ── NOMINAL: evaluate transitions in priority order ──

        # C: Phase Promotion — energy below target OR step budget exhausted, deepen training
        _ph = self.phases[self.current_phase]
        _step_budget = _ph.get('max_steps')
        _step_promotion = _step_budget is not None and step >= _step_budget
        if (avg_e < _ph['target_e'] or _step_promotion) and self.current_phase < 3:
            self.current_phase += 1
            ph = self.phases[self.current_phase]
            for layer in self.model.layers:
                layer.base_local_lr = ph['local_lr']
            self._set_lr(ph['lr'])
            rolling_e.clear(); rolling_l.clear()
            self._enter(self.PROMOTING, step)
            action = self.PROMOTING
            log    = {'surgery': 3, 'phase': self.current_phase}
            print(f'\nPROMOTION [{step}] -> Phase {self.current_phase} ({ph["name"]}) | LR:{ph["lr"]:.2e} | Iters:{ph["iters"]}')

        # B: Noise Wall — high energy, flat loss, not in Phase 3
        elif avg_e > self.phases[self.current_phase]['noise_e'] and l_delta < 0.001 and len(rolling_e) >= 1000 and self.current_phase < 3:
            new_lr = min(self.lr_ceil, self._base_lr() * 1.5)
            self._set_lr(new_lr)
            self._enter(self.NOISE_WALL, step)
            action = self.NOISE_WALL
            log    = {'surgery': 2, 'surgery_lr': new_lr}
            print(f'\nSURGERY B [{step}]: Noise Wall (E:{avg_e:.1f} L_delta:{l_delta:.4f}) LR->{new_lr:.2e}')

        # D: Syntactic Plateau — converged energy, flat loss (Phase 3 only)
        elif l_delta < 0.005 and avg_e < 200 and self.current_phase == 3 and len(rolling_e) >= 500:
            for layer in self.model.layers:
                layer.spectral_gate.spectral_blend.data.add_(0.05).clamp_(max=self.blend_cap)
            new_blend = self.model.layers[0].spectral_gate.spectral_blend.item()
            self._enter(self.PLATEAU, step)
            action = self.PLATEAU
            log    = {'surgery': 4, 'blend': new_blend}
            print(f'\nSURGERY D [{step}]: Plateau (L_delta:{l_delta:.4f} E:{avg_e:.1f}) Blend->{new_blend:.2f}')

        return self.current_phase, action, log


def save_checkpoint(model, opt, step, path, phase, full=True):
    # Disk guard: abort if < 8 GB free to prevent a corrupt half-written checkpoint
    import shutil as _shutil
    free_gb = _shutil.disk_usage('/kaggle/working').free / 1024**3
    if free_gb < 8.0:
        print(f'WARN: Only {free_gb:.1f} GB free — skipping checkpoint at step {step}.')
        return

    gc.collect(); torch.cuda.empty_cache()
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    state = {k: v.cpu().half() for k, v in raw_model.state_dict().items()}
    payload = {'step': step, 'phase': phase, 'model_state': state}
    if full:
        payload['opt_state'] = opt.state_dict()
    torch.save(payload, path + '.tmp')
    os.replace(path + '.tmp', path)
    wandb.save(path, base_path="/kaggle/working/", policy="now")
    print(f'Checkpoint saved: Step {step} (Phase {phase}) | disk_free:{free_gb:.1f}GB | opt:{"yes" if full else "no"}')

def _inject_wandb_id(path, wandb_id):
    """Persist wandb_id in a tiny sidecar file — avoids reloading the 6.4 GB checkpoint."""
    try:
        import json as _json
        sidecar = path + '.meta'
        with open(sidecar, 'w') as f: _json.dump({'wandb_id': wandb_id}, f)
    except Exception as e:
        print(f'Could not persist wandb_id: {e}')

def run_mini_audit(model, tokenizer, step):
    gc.collect(); torch.cuda.empty_cache()
    prompt = "The fundamental principles of biology include"
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"].to(model.device0)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=30, local_iters=16)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'\nAudit (Step {step}): {text}')
    wandb.log({"audit_sample": wandb.Html(f"<b>Step {step}:</b> {text}")}, step=step)

def train_automated(model, tokenizer, start_step=0, start_phase=0, opt_state=None):
    gc.collect(); torch.cuda.empty_cache()
    dataloader = get_dataloader(tokenizer)

    # Resume the same wandb run if a run_id was saved in the checkpoint sidecar.
    _wandb_id = None
    _sidecar = SAVE_PATH + '.meta'
    if os.path.exists(_sidecar):
        try:
            import json as _json
            with open(_sidecar) as f: _wandb_id = _json.load(f).get('wandb_id')
        except Exception:
            pass
    run = wandb.init(
        project='ppc-ocns-v3',
        name=f'pilot-{time.strftime("%H%M")}' if _wandb_id is None else None,
        id=_wandb_id,
        resume='allow',
    )

    PHASES = {
        # target_e: Anderson res_norm target (geometric: 1 OOM per phase)
        # noise_e:  NOISE_WALL absolute trigger (above observed idle baseline ~3800)
        # max_steps: step-based promotion fallback if energy descent is slow post-fix
        0: {'name': 'Init',        'lr': 2e-4,   'iters': 12, 'local_lr': 0.05, 'target_e': 100.0, 'noise_e': 5500, 'max_steps': 3000},
        1: {'name': 'Accel',       'lr': 1.5e-4, 'iters': 20, 'local_lr': 0.15, 'target_e': 10.0,  'noise_e': 4500, 'max_steps': 5000},
        2: {'name': 'DeepConv',    'lr': 1e-4,   'iters': 32, 'local_lr': 0.25, 'target_e': 1.0,   'noise_e': 500,  'max_steps': 8000},
        3: {'name': 'Crystallize', 'lr': 5e-5,   'iters': 48, 'local_lr': 0.10, 'target_e': 0.1,   'noise_e': 50,   'max_steps': None},
    }
    LR_FLOOR  = 1e-6
    LR_CEIL   = 5e-4
    BLEND_CAP = 1.0

    def get_opt(p_idx):
        lr = PHASES[p_idx]['lr']
        spec_params = [p for n, p in model.named_parameters() if 'spectral' in n]
        base_params  = [p for n, p in model.named_parameters() if 'spectral' not in n]
        return bnb.PagedAdamW8bit([{'params': base_params, 'lr': lr}, {'params': spec_params, 'lr': lr*10}])

    opt = get_opt(start_phase)
    if opt_state is not None:
        print("Restoring optimizer momentum from checkpoint...")
        try:
            opt.load_state_dict(opt_state)
            for param, state in opt.state.items():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(param.device)
        except Exception as e: print(f"Could not restore optimizer state: {e}")

    fsm = PilotFSM(model, opt, PHASES, LR_FLOOR, LR_CEIL, BLEND_CAP)
    fsm.current_phase = start_phase

    rolling_e = []
    rolling_l = []
    step = start_step
    print(f'Pilot Engaged: Phase {start_phase} ({PHASES[start_phase]["name"]}) | Resuming from step {start_step}')

    try:
        for step, batch in enumerate(itertools.islice(dataloader, start_step, None), start=start_step):
            ids, targets = batch[:, :-1], batch[:, 1:].to(model.device1)
            opt.zero_grad()
            t_loop = time.time()
            with torch.amp.autocast('cuda'):
                current_rolling_e = sum(rolling_e[-10:]) / 10 if len(rolling_e) >= 10 else None
                logits, avg_iters, avg_energy, _ = model(
                    ids,
                    local_iters=PHASES[fsm.current_phase]['iters'],
                    rolling_energy=current_rolling_e,
                )
                loss = F.cross_entropy(logits.reshape(-1, 128256), targets.reshape(-1))

            loss_val   = loss.item()
            energy_val = avg_energy.item()
            if not math.isfinite(energy_val): energy_val = 0.0

            # Guard 1: Non-finite loss — skip to prevent NaN gradient poisoning.
            if not math.isfinite(loss_val):
                print(f'Non-finite loss at step {step} ({loss_val}). Skipping batch.')
                opt.zero_grad()
                continue

            # Guard 2: Loss spike — skip outlier batches (garbage HTML, encoding errors).
            if len(rolling_l) >= 10 and loss_val > (sum(rolling_l[-10:]) / 10) * 5.0 and step > start_step + 100:
                print(f'Loss spike at step {step}: {loss_val:.3f} vs avg {sum(rolling_l[-10:])/10:.3f}. Skipping.')
                opt.zero_grad()
                continue

            ppl = math.exp(min(loss_val, 20))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step_time = time.time() - t_loop
            rolling_e.append(energy_val)
            rolling_l.append(loss_val)

            # --- FSM TICK ---
            if len(rolling_e) >= 50:
                avg_e   = sum(rolling_e[-50:]) / 50
                e_denom = max(1e-8, sum(rolling_e[-50:-40]) / 10)
                e_trend = (sum(rolling_e[-10:]) / 10) / e_denom
                avg_l   = sum(rolling_l[-50:]) / 50
                l_delta = abs(avg_l - (sum(rolling_l[-100:-50]) / 50)) if len(rolling_l) >= 100 else 1.0

                current_phase, action, surgery_log = fsm.tick(
                    step, avg_e, e_trend, l_delta, rolling_e, rolling_l
                )
                if surgery_log:
                    wandb.log({**surgery_log, 'fsm_state': fsm.state}, step=step)

            if step % 5 == 0:
                s_blend = model.layers[0].spectral_gate.spectral_blend.item()
                print(f'P{fsm.current_phase}|{fsm.state[:3]} | S{step} | L:{loss_val:.4f} | P:{ppl:.1f} | E:{energy_val:.4f} | B:{s_blend:.8f} | I:{avg_iters:.1f} | T:{step_time:.2f}s')
                wandb.log({
                    'loss': loss_val, 'energy': energy_val, 'ppl': ppl,
                    'phase': fsm.current_phase, 's_blend': s_blend,
                    'avg_iters': avg_iters, 'fsm_state': fsm.state,
                }, step=step)

            if step % 50 == 0: gc.collect(); torch.cuda.empty_cache()
            if (step + 1) % 500 == 0: run_mini_audit(model, tokenizer, step+1)
            if (step + 1) % 100 == 0:
                save_checkpoint(model, opt, step+1, SAVE_PATH, fsm.current_phase, full=False)
                _inject_wandb_id(SAVE_PATH, run.id)

    except KeyboardInterrupt:
        print(f'\nInterrupted at step {step}. Saving emergency checkpoint...')
        save_checkpoint(model, opt, step, SAVE_PATH, fsm.current_phase)
        _inject_wandb_id(SAVE_PATH, run.id)
        print('Emergency checkpoint saved. Re-run Cell 5 + Cell 8 to resume.')

    finally:
        wandb.finish()

# 📈 8. Training Phases


In [ ]:
# Engage Automated Pilot
train_automated(model, tokenizer, start_step=START_STEP, start_phase=START_PHASE, opt_state=OPT_STATE)


# 🧠 9. Instant Verification


In [ ]:
# ── Cell 9: 🗣️ Verification (Instant) ────────────
gc.collect(); torch.cuda.empty_cache()
def verify_generation(model, tokenizer, prompt="The fundamental principles of biology include"):
    # --- Atomic Audit Override: Forcing Maximum Inference Precision ---
    for layer in model.layers:
        layer.exit_threshold.data.fill_(0.00005)
        layer.min_iters = 8

    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    print("\n🧠 Regular Generation...")
    t0 = time.time()
    with torch.no_grad():
        out_reg = model.generate(inputs, max_new_tokens=80, local_iters=64)
    print(f"[Reg] {time.time()-t0:.1f}s: {tokenizer.decode(out_reg[0], skip_special_tokens=True)}")
    
    print("\n🐝 Advanced Swarm Inference (N=8)...")
    t0 = time.time()
    # Picking the most resonant ghost state (lowest energy resonance)
    with torch.no_grad():
        out_swarm = model.generate_swarm(inputs, max_new_tokens=80, swarm_size=4, local_iters=64)
    print(f"[Swarm] {time.time()-t0:.1f}s: {tokenizer.decode(out_swarm[0], skip_special_tokens=True)}")
    print("\n🧬 Running MoE Health Audit...")
    captured_indices = []
    def hook_fn(module, input, output):
        captured_indices.append(output[1].detach().cpu())
        
    hooks = []
    try:
        for name, module in model.named_modules():
            if module.__class__.__name__ == 'ExpertChoiceMoEMatcher':
                hooks.append(module.register_forward_hook(hook_fn))
                
        with torch.no_grad(), torch.amp.autocast('cuda'):
            model(inputs, local_iters=8)
            
    finally:
        for h in hooks: h.remove()
    
    B_T = inputs.shape[1]
    total_coverage = 0
    for indices in captured_indices[:24]:
        flat_indices = indices.flatten()
        unique_tokens = torch.unique(flat_indices).shape[0]
        total_coverage += (unique_tokens / float(B_T))
            
    avg_cov = total_coverage / len(captured_indices[:24])
    print(f"🔬 Average Expert Diversity Score (Coverage): {avg_cov:.1%}")
    if avg_cov < 0.3:
        print("🚨 WARNING: High Expert Cloning detected. Experts are picking the same tokens.")
    elif avg_cov > 0.8:
        print("✅ HEALTHY: Maximum MoE utilization. Experts are highly specialized.")
    else:
        print("⚠️ MODERATE: Experts are learning, but there is noticeable overlap.")

# To run: 
# verify_generation(model, tokenizer)


# 🗣️ 10. Generation Sandbox


In [ ]:
# ── Cell 10: 🧪 Generation Sandbox (Verification) ────────────
gc.collect(); torch.cuda.empty_cache()
def run_sandbox_verification(prompt='The fundamental principles of biology include'):
    # --- Atomic Audit Override: Forcing Maximum Inference Precision ---
    for layer in model.layers:
        layer.exit_threshold.data.fill_(0.00005)
        layer.min_iters = 8

    inputs = tokenizer(prompt, return_tensors='pt')['input_ids']
    print('\n🧠 Running Native Generation...')
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=80, local_iters=64)
    print(f'Output:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}')

# run_sandbox_verification()
